# Отчёт по ML-исследованию для защиты

Этот notebook — компактный презентационный отчёт по ML-блоку Game Intelligence Platform.

Перед запуском рекомендуется выполнить:

```bash
make er-merge-strategy-comparison
make er-graph-analysis
make er-embedding-research
make igdb-matching-analysis
make ml-research-defense
make bayesian-rating
make rag-explanations
make ml-defense-readiness
```

Для полного пересбора перед защитой используйте `make ml-defense-all`.


In [ ]:
import csv
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
REPORT_DIR = ROOT / 'data' / 'artifacts' / 'reports'
DEFENSE_DIR = REPORT_DIR / 'ml_research_defense'
ER_DIR = REPORT_DIR / 'entity_resolution'
GRAPH_DIR = REPORT_DIR / 'graph_analysis'
EMBEDDING_DIR = REPORT_DIR / 'embedding_research'
IGDB_DIR = REPORT_DIR / 'igdb_matching'
BAYESIAN_DIR = REPORT_DIR / 'bayesian_rating'
RAG_DIR = REPORT_DIR / 'rag_explanations'
READINESS_DIR = REPORT_DIR / 'ml_defense_readiness'

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_csv(path):
    with path.open(encoding='utf-8') as file:
        return list(csv.DictReader(file))

summary = read_json(DEFENSE_DIR / 'ml_research_defense_summary.json')
merge = read_json(ER_DIR / 'merge_strategies' / 'merge_strategy_comparison.json')
graph = read_json(GRAPH_DIR / 'graph_analysis_summary.json')
embedding = read_json(EMBEDDING_DIR / 'embedding_research_summary.json')
igdb = read_json(IGDB_DIR / 'igdb_matching_summary.json')
bayesian_summary = read_json(BAYESIAN_DIR / 'bayesian_rating_summary.json')
rag_summary = read_json(RAG_DIR / 'rag_explanation_summary.json')
readiness_summary = read_json(READINESS_DIR / 'ml_defense_readiness_summary.json')
summary.keys(), merge.keys(), graph.keys(), embedding.keys(), igdb.keys(), bayesian_summary.keys(), rag_summary.keys(), readiness_summary.keys()


## 1. Базовое состояние данных и ручной разметки

Этот раздел показывает, что в проекте достаточно данных и ручных меток для сильной объяснимой базовой модели.

In [ ]:
summary['baseline_counts']


In [ ]:
summary['training_dataset']


## 2. Влияние качества данных

Артефакты контроля качества данных объясняют, почему Entity Resolution требует ручной проверки и консервативных threshold-правил.

In [ ]:
summary['data_quality']


## 3. Метрики Entity Resolution

Ключевой тезис защиты: модель полезна как слой скоринга, ручной проверки и контролируемого гибридного объединения, а не как слепой автоматический canonical merge.

In [ ]:
summary['existing_er_artifacts']['v3c_metrics']


In [ ]:
read_csv(ER_DIR / 'iterations' / 'manual_threshold_eval_v3c.csv')


## 4. Исследование важности признаков и калибровки

Ablation study показывает, какие группы признаков наиболее важны. Calibration проверяет, можно ли использовать вероятностные оценки модели для threshold policy.

In [ ]:
read_csv(DEFENSE_DIR / 'ablation_study.csv')


In [ ]:
summary['calibration']


### Графики для презентации

![F1 в исследовании важности признаков](../data/artifacts/reports/ml_research_defense/charts/ablation_f1.svg)

![Калибровка вероятностей](../data/artifacts/reports/ml_research_defense/charts/calibration_bins.svg)

## 5. Сравнение стратегий объединения

Теневое сравнение стратегий показывает, почему автоматическое объединение только по модели требует governance-контроля и не должно напрямую заменять доверенный canonical layer.

In [ ]:
merge['strategies']


![F1 по стратегиям объединения](../data/artifacts/reports/ml_research_defense/charts/merge_strategy_f1.svg)

## 6. Анализ ER-графа рисков

Анализ графа рисков показывает транзитивный риск merge-ошибок: качество на уровне отдельных пар может выглядеть сильным, но политика объединения всё равно может создавать компоненты с дубликатами из одного источника или рискованные кластеры.

In [ ]:
graph['strategies']


In [ ]:
read_csv(GRAPH_DIR / 'risky_components.csv')[:20]


In [ ]:
read_csv(GRAPH_DIR / 'high_probability_reviewed_negatives.csv')[:20]


![Дублирующие связи из одного источника](../data/artifacts/reports/graph_analysis/same_source_duplicate_links.svg)

## 7. Лёгкое исследование эмбеддингов названий

Локальный TF-IDF/SVD эксперимент сравнивает векторную близость названий с fuzzy-сходством названий до добавления более тяжёлых нейросетевых эмбеддингов.

In [ ]:
embedding


In [ ]:
read_csv(EMBEDDING_DIR / 'embedding_model_comparison.csv')


In [ ]:
read_csv(EMBEDDING_DIR / 'embedding_error_cases.csv')[:20]


![F1 embedding-моделей](../data/artifacts/reports/embedding_research/embedding_model_f1.svg)

## 8. Анализ сопоставления через IGDB

Раздел оценивает IGDB как источник поисковых ER-кандидатов: качество retrieval rank, точность проверенных пар, рискованные отрицательные примеры и покрытие enrichment-полей.

In [ ]:
igdb


In [ ]:
read_csv(IGDB_DIR / 'igdb_review_precision_by_rank.csv')


In [ ]:
read_csv(IGDB_DIR / 'igdb_enrichment_coverage.csv')


In [ ]:
read_csv(IGDB_DIR / 'igdb_high_risk_reviewed_negatives.csv')[:20]


![Распределение IGDB rank](../data/artifacts/reports/igdb_matching/igdb_rank_distribution.svg)

![Покрытие IGDB enrichment](../data/artifacts/reports/igdb_matching/igdb_enrichment_coverage.svg)

## 9. Активное обучение и примеры рекомендаций

Эти примеры полезны для живой защиты: какие пары модель предложила бы проверить следующими и как базовые рекомендации объясняют общие признаки.

In [ ]:
read_csv(DEFENSE_DIR / 'active_learning_candidates.csv')[:20]


In [ ]:
read_csv(DEFENSE_DIR / 'recommendation_examples.csv')[:20]


## 10. Набор кейсов для защиты и распределение recommendation score

Используйте эти строки как сценарий защиты: успешные объединения, отклонённые рискованные совпадения, пары для активного обучения и примеры рекомендаций.

In [ ]:
read_csv(DEFENSE_DIR / 'defense_demo_cases.csv')


In [ ]:
read_csv(DEFENSE_DIR / 'recommendation_score_distribution.csv')


![Распределение recommendation score](../data/artifacts/reports/ml_research_defense/charts/recommendation_score_distribution.svg)

## 11. Grounded RAG-like explanations

Раздел показывает русскоязычные объяснения, сгенерированные только из рассчитанных фактов. LLM/RAG-like слой не принимает решения по совпадениям или рекомендациям.

In [ ]:
rag_summary


In [ ]:
read_csv(RAG_DIR / 'match_explanation_examples.csv')[:10]


In [ ]:
read_csv(RAG_DIR / 'recommendation_explanation_examples.csv')[:10]


In [ ]:
read_csv(RAG_DIR / 'grounded_fact_cards.csv')[:10]


## 12. Анализ Bayesian rating

Bayesian rating — дополнительный статистический блок: он сравнивает наивные рейтинги источников с рейтингами, скорректированными по числу голосов, и показывает, почему игры с малым числом голосов нужно сдвигать к global mean.

In [ ]:
bayesian_summary


In [ ]:
read_csv(BAYESIAN_DIR / 'canonical_bayesian_ratings.csv')[:20]


In [ ]:
read_csv(BAYESIAN_DIR / 'low_vote_shrinkage_examples.csv')[:20]


![Лучшие Bayesian ratings](../data/artifacts/reports/bayesian_rating/top_bayesian_ratings.svg)

## 13. Финальный readiness gate перед защитой

Этот раздел проверяет, что все обязательные исследовательские артефакты и ключевые метрики доступны перед репетицией защиты.

In [ ]:
readiness_summary


In [ ]:
read_csv(READINESS_DIR / 'ml_defense_metric_snapshot.csv')


In [ ]:
read_csv(READINESS_DIR / 'ml_defense_demo_sequence.csv')


In [ ]:
read_csv(READINESS_DIR / 'ml_defense_artifact_checklist.csv')


## 14. Выводы для защиты

- Entity Resolution — самый сильный текущий ML-исследовательский блок.
- Ручные метки критичны: они дают и положительные, и отрицательные примеры.
- `model_auto_090` полезен для кандидатов с высокой уверенностью, но не как слепой canonical merge.
- `model_auto_070_research` полезен для анализа рисков, потому что выявляет ложноположительные совпадения и проблемы кластеров.
- Анализ графа рисков показывает транзитивный ER-риск, который не виден только по метрикам отдельных пар.
- Лёгкие эмбеддинги названий улучшают fuzzy-only baseline без скачивания внешних моделей.
- Анализ IGDB rank показывает, почему поисковые кандидаты требуют governance-контроля retrieval и выборочной ручной проверки.
- Рекомендации — объяснимый content-based baseline, его стоит показывать как вторичный ML-блок.
- Grounded RAG-like explanations только визуализируют рассчитанные факты; они не принимают решения по объединениям или рекомендациям.
- Bayesian rating демонстрирует отдельный интерпретируемый статистический блок для устойчивого ранжирования при sparse vote counts.